In [1]:
# Jupyter Notebook: JobHunter Backend Execution

# Cell 1: Setup
import os
import sqlite3
import pandas as pd
import time
import webbrowser
from jobhunter import config
from jobhunter.dataTransformer import DataTransformer
from jobhunter.extract import extract
from jobhunter.FileHandler import FileHandler
from jobhunter.load import load
from jobhunter.SQLiteHandler import (
    save_text_to_db,
    update_similarity_in_db,
    fetch_resumes_from_db,
    delete_resume_in_db
)

RAW_DATA_PATH = config.RAW_DATA_PATH
PROCESSED_DATA_PATH = config.PROCESSED_DATA_PATH
RESUME_PATH = config.RESUME_PATH
DATABASE_PATH = config.DATABASE

# Cell 2: Initialize Database
def initialize_database():
    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS jobs_new (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            primary_key TEXT UNIQUE,
            date TEXT,
            resume_similarity REAL DEFAULT 0,
            title TEXT,
            company TEXT,
            company_url TEXT,
            company_type TEXT,
            job_type TEXT,
            job_is_remote TEXT,
            job_apply_link TEXT,
            job_offer_expiration_date TEXT,
            salary_low REAL,
            salary_high REAL,
            salary_currency TEXT,
            salary_period TEXT,
            job_benefits TEXT,
            city TEXT,
            state TEXT,
            country TEXT,
            apply_options TEXT,
            required_skills TEXT,
            required_experience TEXT,
            required_education TEXT,
            description TEXT,
            highlights TEXT,
            embeddings TEXT
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS resumes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            resume_name TEXT UNIQUE,
            resume_text TEXT
        )
    ''')

    conn.commit()
    conn.close()

initialize_database()
print("✅ Database initialized")

# Cell 3: Upload Resume File
resume_name = "sample_resume.txt"
resume_text = open(resume_name, encoding="utf-8").read()
save_text_to_db(resume_name, resume_text)
print(f"✅ Resume '{resume_name}' uploaded and saved to DB")

# Cell 4: Run Extraction, Transformation & Loading
job_titles = ["Data Scientist", "ML Engineer"]
country = "IND"
date_posted = "week"
location = "Remote"

print("🔍 Extracting job data...")
total_jobs = extract(job_titles, country=country, date_posted=date_posted, location=location)
print(f"✅ Extracted {total_jobs} jobs")

print("🧠 Transforming job data...")
file_handler = FileHandler(raw_path=RAW_DATA_PATH, processed_path=PROCESSED_DATA_PATH)
DataTransformer(
    raw_path=RAW_DATA_PATH,
    processed_path=PROCESSED_DATA_PATH,
    resume_path=RESUME_PATH,
    data=file_handler.import_job_data_from_dir(RAW_DATA_PATH)
).transform()

print("💾 Loading jobs into database...")
load()
print("✅ Jobs loaded into database")

# Cell 5: Calculate Resume Similarity
update_similarity_in_db(resume_name)
print("✅ Similarity scores updated")

# Cell 6: View Top Matches
conn = sqlite3.connect(DATABASE_PATH)
query = """
    SELECT 
        title, company, resume_similarity, job_apply_link
    FROM jobs_new
    ORDER BY resume_similarity DESC
    LIMIT 10
"""
df_top = pd.read_sql(query, conn)
conn.close()

df_top

# Cell 7: Open Top 10 Apply Links (Optional)
for link in df_top["job_apply_link"].dropna():
    if isinstance(link, str) and link.startswith("http"):
        webbrowser.open_new_tab(link)
        time.sleep(0.5)


ModuleNotFoundError: No module named 'jobhunter'